# Kinematic magnetic winding

In the ideal fixed-flow limit, $q(t)=t\,\mathbf B_p\cdot\nabla\Omega$ for $q(0)=0$, where $q=B_\phi/(r\sin\theta)$. The shell spans $0.3\le r\le1$.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
from nsinduct.shell import (
    initial_omega,
    make_grid,
    poloidal_field,
    run_case,
    volume_rms,
    winding_operator,
)

base = dict(
    ntheta=16,
    nr=24,
    r_inner=0.3,
    r_outer=1.0,
    field_strength=0.07,
    eta=0.0,
    nu=0.0,
    timestep=0.01,
    output_dt=0.1,
    snapshot_count=3,
    stop_time=12.0,
    lorentz_feedback=True,
)

In [ ]:
cfg = base | dict(stop_time=2.0, lorentz_feedback=False, name="winding")
result = run_case(cfg)
g = make_grid(cfg)
p = poloidal_field(g, cfg["field_strength"])
o = initial_omega(g, cfg)
source = winding_operator(g, p["br"], p["btheta"], o)
q = result["snapshots"][-1]["fields"]["q"]
relative_error = volume_rms(g, q - 2 * source) / volume_rms(g, 2 * source)
print(f"Error against discrete exact winding: {relative_error:.3e}")
assert relative_error < 1e-11
rows = result["rows"]
t = np.array([r["time"] for r in rows])
rms = np.array([r["rms_B_phi"] for r in rows])
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8), constrained_layout=True)
ax[0].plot(t, rms, label="numerical")
ax[0].plot(t, t * volume_rms(g, g.s * source), "--", label="linear winding")
ax[0].set(xlabel=r"$t$", ylabel="RMS toroidal field")
ax[0].legend()
theta_edge = np.linspace(0, np.pi, g.ntheta + 1)
r_edge = np.linspace(g.r_inner, g.r_outer, g.nr + 1)
rr, th = np.meshgrid(r_edge, theta_edge)
im = ax[1].pcolormesh(
    rr * np.cos(th), rr * np.sin(th), g.s * q, shading="flat", cmap="RdBu_r"
)
ax[1].set(xlabel="z", ylabel="s", title="Meridional toroidal field at t=2")
fig.colorbar(im, ax=ax[1], label=r"$B_\phi$")
plt.show()

The finite-difference comparison uses its discrete winding source and checks time integration.

## Continuum winding with Dedalus

Evaluate the analytical source on the native spectral nodes.

In [ ]:
import logging

logging.getLogger("dedalus").setLevel(logging.WARNING)

from nsinduct.dedalus_backend import run_case as run_spectral, normalization

spectral_config = dict(ntheta=16, nr=24, field_strength=0.07, eta=0.0,
                       nu=0.0, lorentz_feedback=False, stop_time=0.2,
                       timestep=0.005, output_dt=0.02, snapshot_count=2)
spectral = run_spectral(spectral_config)
snapshot = spectral["snapshots"][-1]
f = snapshot["fields"]
r, theta = np.meshgrid(f["radius"], f["theta"])
norm = normalization(spectral_config)
h = ((r - 0.3) * (1.0 - r))**2
hp = 2*(r - 0.3)*(1.0 - r)*(1.3 - 2*r)
br = 2*norm["poloidal_scale"]*h*np.cos(theta)/r**2
bt = -norm["poloidal_scale"]*hp*np.sin(theta)/r
k = 2*np.pi/0.7
radial = 1 + 0.18*np.cos(k*(r - 0.3))
p2 = (3*np.cos(theta)**2 - 1)/2
omega_r = -0.18*k*np.sin(k*(r - 0.3))*p2/norm["omega_amplitude"]
omega_theta = -3*radial*np.sin(theta)*np.cos(theta)/norm["omega_amplitude"]
q_exact = snapshot["time"]*(br*omega_r + bt*omega_theta/r)
relative_max_error = np.max(abs(f["q"] - q_exact))/np.max(abs(q_exact))
print("Dedalus continuum winding relative maximum error:", relative_max_error)
assert relative_max_error < 1e-6
j = len(f["radius"])//2
plt.plot(f["theta"], f["q"][:, j], label="Dedalus")
plt.plot(f["theta"], q_exact[:, j], "--", label="analytic winding")
plt.xlabel(r"Colatitude $\theta$ (rad)")
plt.ylabel(r"$q$ at fixed radius")
plt.legend()
plt.show()
